# 第 2 天练习 —— 用本地 Ollama 做网页摘要

## 练习目标（理念）

把 **Day 1** 的「网页抓取 + LLM 摘要」升级为走 **本地 Ollama**，而不是付费的 OpenAI API。

- **输入**：一个网页 URL
- **输出**：模型生成的 Markdown 短摘要
- **额外要求**：任选一种调用方式（`requests` / `ollama` 包 / OpenAI 兼容客户端）

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 `requests` + BeautifulSoup | `Website` 类清洗 HTML |
| Chat Completions / messages | `system` + `user` 组成 `messages` |
| 本地开源模型 | `MODEL = "llama3.2"` |
| 三种调 Ollama 的方式 | 原生 HTTP、`ollama.chat`、`OpenAI(base_url=...)` |

## 怎么跑

1. 确保本机 Ollama 已安装并在 [http://localhost:11434/](http://localhost:11434/) 显示 running
2. `ollama pull llama3.2`（机器慢可用 `llama3.2:1b`）
3. 从上到下运行；练习区把 URL 换成你想摘要的网站


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="宽度：150px；高度：150px；垂直对齐：中间；">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">开始作业前 —— 课程资源页</h2>
            <span style="color:#f71;">这里汇总了幻灯片与常用链接，建议先收藏：<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            后续还会持续补充。
            </span>
        </td>
    </tr>
</表>


# 家庭作业：把 Day 1 网页摘要改成走本地 Ollama

目标：升级第 1 天项目，用通过 **Ollama** 在本地运行的开源模型做网页摘要，而不是 OpenAI。

若不想用付费 API，后续项目也能复用这套本地调用方式。

**好处：**
1. 无 API 收费——开源模型本地跑
2. 数据不离开你的机器

**缺点：**
1. 能力通常弱于 Frontier（前沿）商业模型

## Ollama 安装回顾

1. 打开 [ollama.com](https://ollama.com) 安装
2. 安装后本机应已有 ollama 服务；访问 [http://localhost:11434/](http://localhost:11434/) 应看到 “Ollama is running”（或中文界面等价提示）
3. 若没有：新开终端执行 `ollama serve`；另一终端执行 `ollama pull llama3.2`
4. 机器较慢时，可改用 `llama3.2:1b`：`ollama pull llama3.2:1b`，并把代码里的 `MODEL = "llama3.2"` 改为 `MODEL = "llama3.2:1b"`


In [ ]:
# ========== 导入：网页请求、HTML 解析、笔记本展示 ==========

# 导入 requests：用 HTTP 向本地 Ollama API 发 POST
import requests
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可抓取的文档树（后面练习会用到）
from bs4 import BeautifulSoup
# 从 IPython.display 导入 Markdown、display：在笔记本里漂亮显示 Markdown
from IPython.display import Markdown, display


In [ ]:
# ========== 常量：Ollama 地址、请求头、模型名集中配置 ==========

# Ollama 原生聊天 API 地址（/api/chat，不是 OpenAI 兼容的 /v1）
OLLAMA_API = "http://localhost:11434/api/chat"
# HTTP 头：声明请求体是 JSON
HEADERS = {"Content-Type": "application/json"}
# 本地模型名：需事先 ollama pull；字符串必须和本机已安装模型一致
MODEL = "llama3.2"


In [ ]:
# ========== messages：与 OpenAI Chat Completions 相同的 role/content 列表 ==========

# 使用与 OpenAI 相同的格式创建消息列表；发给模型的 content 保持英文（影响回答行为，不翻译）
messages = [
    {"role": "user", "content": "Describe some of the business applications of Generative AI"}
]


In [ ]:
# ========== payload：组装发给 Ollama /api/chat 的 JSON 请求体 ==========

# model：用哪个本地模型；messages：对话；stream=False：一次性返回完整结果（非流式）
payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False
    }


In [ ]:
# ========== 拉取模型：确保本机已有 llama3.2（魔法命令 ! 走 shell）==========

# 若已拉取过会很快结束；首次可能下载几分钟
!ollama pull llama3.2


In [ ]:
# ========== 方式一：用 requests 直接 POST 到 Ollama 原生 API ==========

# 若本格失败，可试后面两种方式，并复查顶部「Ollama 安装回顾」
# 若仍不行，按课程说明排查或提问

# POST：把 payload 以 JSON 发给 OLLAMA_API；headers 声明 Content-Type
response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
# 响应 JSON 里 message.content 是助手回复正文
print(response.json()['message']['content'])


## 介绍 `ollama` Python 包

接下来做同样的事，但改用更简洁的 **ollama** Python 包，而不是手写 HTTP。

底层仍会请求本机 `localhost:11434` 上的 Ollama 服务，与上一格等价。


In [ ]:
# ========== 方式二：用 ollama 包封装同一调用 ==========

# 导入 ollama：官方/社区 Python 客户端，封装 /api/chat
import ollama

# chat：指定 model 与 messages；返回结构里仍有 message.content
response = ollama.chat(model=MODEL, messages=messages)
# 打印助手回复
print(response['message']['content'])


## 替代方法 —— 用 OpenAI Python 库连接 Ollama

第三种常见写法：把 Ollama 的 **OpenAI 兼容端点**（`/v1`）当作 `base_url`，复用 `OpenAI` 客户端。


In [ ]:
# ========== 方式三：OpenAI 客户端 + base_url 指向本地 Ollama ==========

# 有人更喜欢这种写法：API 形状与云端 OpenAI 一致，只换地址

# 从 openai 导入 OpenAI 客户端类
from openai import OpenAI
# base_url 指向 Ollama 的 OpenAI 兼容层 /v1；本地常把 api_key 写成任意非空字符串如 'ollama'
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

# 与云端相同的 chat.completions.create 调用；model 仍是本地 MODEL
response = ollama_via_openai.chat.completions.create(
    model=MODEL,
    messages=messages
)

# OpenAI SDK 路径：choices[0].message.content
print(response.choices[0].message.content)


## 为什么能用 OpenAI 代码调用 Ollama？

看起来奇怪：明明写的是 OpenAI 客户端，却在问本地 Ollama？

要点：

- Python 类 `OpenAI` 本质是**客户端库（client library）**：在你电脑上发 HTTP 请求。
- 默认会打到 `https://api.openai.com/v1/chat/completions`；真正算力在 OpenAI 云端。
- 许多厂商提供**同一形状的 Web 端点**，于是可复用同一套客户端。
- Ollama 在本机暴露：`http://localhost:11434/v1/chat/completions`
- 第 2 周还会看到 Gemini、DeepSeek 等也提供兼容端点。

因此：

```python
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
```

只是把「请求发往哪里」从 OpenAI 云改成了本地 Ollama，调用写法不变。


## 再试推理模型 DeepSeek（可选）

这里用精简到约 **1.5B** 的 DeepSeek-reasoner 变体。  
它基于 Qwen 的 1.5B，并用 DeepSeek R1 生成的合成数据微调。

更多尺寸见 [ollama.com/library/deepseek-r1](https://ollama.com/library/deepseek-r1)；完整 671B 体积巨大（约 404GB），多数机器装不下。


In [ ]:
# ========== 拉取 DeepSeek 小模型（首次需下载）==========

!ollama pull deepseek-r1:1.5b


In [ ]:
# ========== 用同一 OpenAI 兼容客户端问 DeepSeek：可能较慢，先看 <think> 思考痕迹 ==========

# 这格可能跑几分钟；回复里常见 <think>...</think> 再给出定义

# 换 model 字符串为 deepseek-r1:1.5b；messages 仍是 user 一条
response = ollama_via_openai.chat.completions.create(
    model="deepseek-r1:1.5b",
    messages=[{"role": "user", "content": "Please give definitions of some core concepts behind LLMs: a neural network, attention and the transformer"}]
)

# 打印完整助手内容（含思考痕迹时一并显示）
print(response.choices[0].message.content)


# 现在是你的练习

把 **Day 1** 的网页抓取与摘要代码合并到这里：用本地 **Llama 3.2**（而不是 OpenAI）做网站摘要。  
上面三种 Ollama 调用方式任选其一即可。


In [ ]:
# ========== 练习区导入：抓网页 + 解析 HTML + 笔记本展示 ==========

# 导入 os：读环境变量（本练习主要走本地 Ollama，未必用到，但常与 Day1 模板一起保留）
import os
# 导入 requests：GET 网页 HTML
import requests
# 从 dotenv 导入 load_dotenv（若后面接云端密钥会用到；此处可保留注释习惯）
from bs4 import BeautifulSoup
# 从 IPython.display 导入 Markdown、display：展示摘要
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI（若改用方式三可取消注释并创建客户端）


In [ ]:
# ========== Website 类：抓取 URL，用 BeautifulSoup 抽出标题与正文 ==========

# 浏览器 User-Agent：部分网站会拒掉默认 python-requests 标识
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        # 保存原始 URL
        self.url = url
        # GET 下载页面；带上伪装浏览器头
        response = requests.get(url, headers=headers)
        # 用 html.parser 解析响应字节为 DOM
        soup = BeautifulSoup(response.content, 'html.parser')
        # 有 <title> 就取字符串，否则占位
        self.title = soup.title.string if soup.title else "No title found"
        # 删掉 script/style/img/input 等对摘要无用的节点
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 从 body 取纯文本：换行分隔并 strip 空白
        self.text = soup.body.get_text(separator="\n", strip=True)


In [ ]:
# ========== 实例化：换成你想摘要的真实网址再跑 ==========

# 构造 Website；下面两行打印标题与清洗后正文，便于检查抓取是否成功
webpage = Website("https://www.pleasurewebsite.com")
print(webpage.title)
print(webpage.text)


In [ ]:
# ========== system_prompt：定角色与输出格式（发给模型的英文指令不翻译）==========

# 反斜杠续行把长字符串拼成一行逻辑；要求 Markdown 短摘要、忽略导航噪音
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."


In [ ]:
# ========== user_prompt_for：把标题 + 正文拼进 user 侧提示 ==========

def user_prompt_for(website):
    # 先告诉模型「你在看的网站标题是什么」
    user_prompt = f"You are looking at a website titled {website.title}"
    # 再要求短摘要；若有新闻/公告也一并概括（英文指令保持原样）
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    # 最后附上抓取到的纯文本正文
    user_prompt += website.text
    return user_prompt


In [ ]:
# ========== messages_for：组装 system + user 两条消息 ==========

def messages_for(website):
    # 返回标准 Chat messages 列表，供 ollama.chat / OpenAI 兼容接口使用
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]


In [ ]:
# ========== 为当前 webpage 生成 messages ==========

# 调用上面函数；结果可直接传给本地模型
messages=messages_for(webpage)


In [ ]:
# ========== 用 ollama 包 + llama3.2 做网站摘要（方式二示例）==========

# 导入 ollama 客户端
import ollama
# 本地模型名；慢机器可改成 "llama3.2:1b"
MODEL = "llama3.2"
# 非流式 chat：把 system/user messages 发给本地模型
response = ollama.chat(model=MODEL, messages=messages)
# 打印摘要正文
print(response['message']['content'])
